# P05 on PolyBench with a T4 GPU (Phase 3.2)

Runs on **Colab** or **Kaggle** (it works in `/content` or `/kaggle/working`, whichever exists).

- **Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Keep the
  tab open; if the session disconnects, reconnect and Run all again: the run resumes from the
  rows already in the CSV (unless the runtime was reset).
- **Kaggle:** File → Import Notebook (this file). In the right-hand panel, Session options:
  **Accelerator → GPU T4 x2** and **Internet on** (both need a phone-verified account). Then
  **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 hours) with
  the browser closed, and the CSV appears under the version's **Output**.

Installing the NVIDIA HPC SDK takes about 10 minutes; the benchmark run takes longer (30
kernels, 6 variants, 3 runs each at the LARGE size).

Variants per kernel: `seq` (nvc -O3, no OpenMP), `seq-mp` (the same source with the OpenMP
flags: what `-mp=gpu` alone does), `p05` (P05's own choices with the T4
machine model: GPU, host threads or sequential per loop), `p05-naive` (every loop P05 can
offload goes to the GPU), and both again as `-warm` (the device started before `main`, so
the timed kernel doesn't pay the ~0.16 s start-up). Correctness: exact hex dumps against
`seq`. `gpu_launches` counts kernel launches (0 = every guard chose the host).

**When it finishes, download `polybench-t4.csv` (Colab: the last cell offers it; Kaggle: the
version's Output) and give it to Claude**; it goes to `eval/polybench/results/t4-large.csv`.

Generated by `eval/polybench/make_gpu_notebook.py` at P05 709f553 (30
kernels, LARGE_DATASET, 80 GPU pragmas in the gated rewrites): edit that and `gpu_run.py`, not this
notebook.


In [ ]:
# Where to work: Colab's /content or Kaggle's /kaggle/working.
import base64, gzip, os
ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.makedirs(ROOT, exist_ok=True)
os.environ['P05_ROOT'] = ROOT  # the %%bash cells cd here
os.chdir(ROOT)
RUNNER = gzip.decompress(base64.b64decode("""
H4sIAAAAAAAC/51aa3PbNhb9rl+BZWYn5EaiZafN7irVzrh+tJnJw+M47XS8Hi5EQhZrimQBypbW
4/++516AFKmHk60/RCQBXFzcx7kP5MVfDhZGH0zS/EDl96JcVbMif93zPO9iJo0Sr8OjkbgYfi+K
XFwU2epHlcezgxPxXXgUHoqHtJoJKX66+CL8kyKTE3H1XV/k97EYzMvxbbkIwl5P4M/RFfgU6UUe
lisxGJQTUYLkhEji1ehYaPWg00oZcT0YYJ4Rr2/oMTb3tDTE702v91ll00Fc5JVMc5UI31RJlk7A
YbYKRkLdy+ygoXswl3cqom3zolKTorijvdV8ohIj0qrfk3lC53tpmr37Is0tlZiOtKYVVd+FabnK
J31h0jxWopopUZPtxTJ/WYk4K3I7oFVZmLQq9Cp0h5sVGTbd3GwuE0XSxZpeou5VVpRzlVf4Hs9w
PivjQf3mfi1rAzD0uynyEWSMDQ5uZaWSgx/ulM5V9q8whuDtbsVDLuJZkcaQrA9t9cUJVFbNtJLg
qNDCqD8W2DSVWeBo5TK9V11aA0gijVeCh0jQSq/E5+PzM5EVRQlpCshAFNNpVsjEHYlso9e7UFpY
SsJ/f3z501l0enx1/Pnsqi8Gn14HzL/6QzR/tLDQ6W2aywz2VIhPpco/XNhpMK0d06yc6IudK6aZ
vMVpsbjU8nYuDWzjYSar2jR7ovsnWXNJARFVBROaFQZHKqAeMpK0MoJOVqX5Lew2zStQKOEZzR+L
v9Zsv+bDycNOHrDsaLJ9cJPd4IPU8/56Gr9vskmMGTkHfet8MNV5Wa1EJfWtqkDwNoXk4TtioqaF
JouBOftykw78x1R6EcNAyd5pqzCGiIh+lc5xEKcwCBCSuYUGS7kybKYbpGC1sCyYmQET1WBRYrtJ
sajEMDx8I0xtCQQNHz9dnX0WR8OjN4PD4eDwTdDfJPYwS2PCFNqTICAnccuyhOVB9CDFXBRwv97P
pB9nxCPx6cNF9PHLh+jq58uz49PPYgyvqy3cgtNIHIl7GL4hVDoptFZxlSuDtVJcfHr/249nH09+
jk6/gNDx5eXxb5/FZJFmsOSpUHA7cS91KuGZpYbyiS21lHHVm6mlNQw4c1zMS6khPFYOjBVS8T2e
55GfkRwy0pQhVWU40T28Pp1OlQbCqCAUV06/rMK4WOQwO8JXp41MLoBE5MYffznBkd+9P7uMjk9O
Ikj23flvb3sEvg4Jh2KuJDDUOmrHQMBWOvUDAgWjGmOHWC6LByNwAhK5yhOcxDnDyedfhGT1r8Q0
zVMzewuxaUVsmru0BKTx0owkvuphnoaNmgKTktTA2nJIG+RsmDAQO9upMos5TvNA0wlBTFVg4ySk
CNRL52WhK7BzC6EaVb8jCtSPhamfdDNsZosqzZq3xaTUBYCvmWlWptc7f3f2no3k2rOS9frCcwqm
R2c59Airrhb8RH4RNU/8EFsz8tp27M3lMoJ2I6U1zaHoU+uNF7MmIgdLTAUzWq+JQmjLvJveL8eX
744/XjGfvgdzolFCPPyey8wo9xP0BQ8D2lozrvRiPaHjZ74HkGHWCLK+PtXiES2oHzoL3CTCkG2i
9O9+ms2iLmFedNP79fjyQ3SC08MaXiDkZguA8Q/wsXD2r14UyarS6WRRqSjy/RagBQFBUZXG4r5I
EwLUiLbx6S0Qj+AFDiyWoDt823thBS9AtfaRuSz9qpjqYj4SSwqIS/FqLA7f9p7YLHuI01PyTz+e
J32BvGn8EbGjz7gJ3OM3DmpCKGySpHHlFybExFQXeWAHwkWZQFA+pV1Ahscn+73Sq1EjLI3VawMO
my1jWVYLrSJsVmI/K7NKLetHYqnFj/sN1nQVlsP7QvtAMa6PNyRSmOaeYLyW0WWsyrYfhVeW3Nmy
TAF1o02qg0Oos3YSzKsF5rDR12raF7dF5SQEib5kgHzZ34eQYo2QiGrVg1I2ouQL5HHahKQUIpVC
K9hoPCbyW3w5GO4jLA15TPbFBALGktAgvlR+wHzVLzXJTOW+DMRfxvw0CbYJW/bIdTkO+F6aTz27
/qHQOMi42RMhWSz7YkUx979p6RMPLYrYbkn8t2zABWsEnMU68nashOmC6HSFjZiDkGwXcclfBv2N
L6u1FTjN/iKzhTrTutBdklt7bO+zpt+m+w20v1F6TiTTJUl/usHOemBJhgOudkz6PzZqqwsI7vNz
X8iJ8bHDAKQDccAj9lPgxlZ4OFSD18NhYIm5/XyvKjKlJazWI2Yt7R/GNPmfQgE51yyBBA/DVeJM
GiO4zrJHId+JIoTcCjBnUPv0KSKaltXQx1CCbfre/VqShQN7SlnNQrBLvz5Nw0jQnQoG7vZNprH1
dEyhsipJtfGbpUCdZWqqqLhjENokDvxtEf8d+XN7qWfzz5Y2CqQffrOUZnhByMmyb6PCxgYAP00i
uL5pvr+A2N59vLq4uoyufrs4iyJOapBJkPPZwuh2IXVi3oqfTk44yT/JJGX3gDU1pXIrRU5UkwP8
TdRzZ+AJ24fgz50DeCnXd1juIlJjMhgI/A2ug+Fb8fTvvOsSFAquWTVx3Bf1FoPCcy/ilfBCAt+b
4Hp4Q54x7DpGV2je4HRj1zEl4EhBGhO0aZJVeMv2ONHlc27LpUSl7FEyllapTX24jp5LfRdlsBUP
gRo55bTLmZPFdYYopAGRwfXR6IaBMyPFTRmT67EWgzMkbMpZZF/cES99m6K32EWoOAaclivK6imG
2GPBFNxy2oK+szfAwhUlFCvhM6FRk/s3BNfZf/AW6e5CU41NCTSFuzRDxptDyi6Lhh3nlI3TJtT/
aCIXa0THO62LpFh/BUM5igOfTxesLSJ5zi5niR6ATQYhPoYDH3z31qQn0iimDT6Cjq3lRdXMSg1Y
8JOgq7I2ICRdRCW9Tem8mEM6p+W0wzZME54jR0oMmZTvhTNvxyQOTHTYbYvTcIRpEISUuPvBzpW1
AHbTrWlXIFFmMlb+S++vw/AomwoPyQleJB6CjdHu4E7C28wmxGoHE7azs2Rt2lyHOsPOufh35s2H
hqH1hUXJTXN/1i4mab5L/3tsbTf8J3thXy23wDKx3AObfG+QLFCsbJqk19rmhfhPk/N7dzCH/wC0
pObql7sg7Gyuu7X21CnYgqM692tRk9bvW+7Y+PwDqj4qOtf5VrnaKbkaYzroAl16j3Swp8HjTm96
amG3rU5D2sGaLD21HFlNjQNkD3KykT1EoQB61VazBIh/urNzsSXY9ryrdx/OvHWYtI0y3vPTa++G
tOO2hQgjHq1TYqJKdRLTvL4Jag45jqylNyez868dlTgmmnaXV/aAr2izdzC+54TKMzZP/NUgE1iJ
/v8LKYDXnV4K4jebFF7VRyJXu+GMjpKalixwqIyLWY7DcICbllND1REdmCoOW8d1A3q8I0g7JLDV
Jbiuo57VR4YMxUApQNhrz7aqppJCDgX9weHNJqCAIarM1rCCMtGBCpTaQg3i8dHbaKh5IxTVujYN
1x8JntpnAJUu/+qaqVxRy/cq+nR+/v7T8SlMbFwb9ro52uXUlYyuJ+IzAlnSVOXW+M+f23hPI1Sk
aRUifUtklvna21HgV0rOSeNEzCFO/LWlqF3xUWUU0bpLHdOoFWLHN+d2jmFZ2sScW1jhsb5dUHv/
gt4QSJWJoVFqbI6jKCniKOroFrlb4KiEMkki6ZYDP5E+gA2t/lhQFe4K/5nKyrG3fVHTwKO3lxzA
aB+9NbhytmevGThftrcE+4lSpAFVyEQusmrs8fve2bG5b09u3JHuOWhs/8K4vS6/j/dPhSoHjEXt
Fe5KQAzwzziO//79M+utxbZXn56dH395f7UNVi5BJBFuuwFnRq37Ctd8NPu3dlGgvXXnMmX/ynU3
s1qVaowUdk3jaO8quoDbueT1/o1cy2fXqjfDYW1QRsUFEj1RIhXGLvsZd0WHR4kDip2x97f9cxsB
bk/WHOCwhp2QVhmHGFQes6twnRs4bGS8zdUDAxX3hcgdtvtz6+LHlmLmfldBQ+QIUX3dNJtv4GnX
Tb8ZwYOMgcsC0AhP07i65KDoT1sIy9dGzBT3XdvduXPA/seiOi8WebLRbimlsT0BupEZd7lFOJAe
n5TwZuzVTSvMq/n4lbJT7XNf0DbNm8YYlrVEYfNYG8t9J0g6FUd1OtkkrEvIVqhpdxwY+ACr/lYS
5VJQoGE73PApHE1GozsuVbBVe+QrnTRuGHKE7bVLFptl2xSzlWET8bopv9WN4l1rK3TwOFcdnrZH
EfS5sb9dkmyx6rbx72waHRBVtq0/S8xWnN3KZMJ5kRU47WZFQD1Je62wC+M4BepQ4cNyetuIcI9O
uxVTBTnHFDDryE/pczclKh7qlrpV8BjicGIdW605tBszE/UVYDtzPezvq/+6VzNjy05zNzOOyy4z
L0TrGpHVQFd19V1P2LW8mYqpSaZ1f32LN+ZO+TAccmLWbaDiJFlxiymT0NZ/X6/8NkqwpuRV2wZB
1w5EmnJAyv52leNb+Vxr8bW38wKSkzvvcNtMNvPfa0qP3U1Ffr++q7BK27ywqP84LWJU7qRJ2/PW
AjaLuX+4bh9ZCq6FJHXlug12gTj5cnrsWkJesE2WalY6379zz9p5toOwR0S82u2znZLdme13LIW2
qaOpWwEfHBxaC556qLor8ajjp21Rq4zAmfBgvBcP1tBHZ9pNgsZTwwchfHyGWTZrChrtKx4iHHwV
vL4Rqv6cO9j7yU0e6EA7fcLFMECMzwCDh76w179jW2ANXIFFWQ3d0Y7B0vXoaIg0fVvPUF04zRZm
tsNAuY/oT73Hu9HhmyfB7YPR4fdPol3HjcQj0z88Gt488bUFiO3w8b1S42vq+gjcHKfb4Ttvq0MX
kTq0zG+VhW5K/XZ03yyg7/biNZR8qzN/zQ8atv+sI3CCB/i/2xphwYT2/zj49jbIFN3q2ucaekOt
z1mI/aH2A4eEsfWMvZ1G+mv9R4ExbAE/o/D17ROf06KAa4s9T6b9HwzGDfS1I97XKNj/2UA8zIFq
LJ1gFL6ZWlb4/dsJwVmEw0cQXDo6fPPJ3XWm/jwZ510eb1/bwZht1/Kx7oVcD+B+o01F7fO9vX73
aHcZ/eNJ7DKkjmQ2ZGLt52Uu85fB6HAYfjd9MkI8sgJH/9xHr1HYY/20y8XpHHFWGEUpdS+lm0Bi
OYpYHFFEnYYocgBv2w69/wHxMJxv1CkAAA==
""")).decode()
open(os.path.join(ROOT, 'gpu_run.py'), 'w').write(RUNNER)
print('working in', ROOT)


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%bash
# PolyBench/C 4.2.1 at the commit eval/llm_baseline/fetch_polybench.sh pins.
set -e
cd "$P05_ROOT"
[ -d polybench/.git ] || git clone -q https://github.com/MatthiasJReisinger/PolyBenchC-4.2.1 polybench
git -C polybench checkout -q 3e872547cef7e5c9909422ef1e6af03cf4e56072
test "$(git -C polybench rev-parse HEAD)" = 3e872547cef7e5c9909422ef1e6af03cf4e56072 && echo "polybench at 3e872547cef7e5c9909422ef1e6af03cf4e56072"


In [ ]:
# P05's rewrites (made with -machine machine/colab-t4.json), packed by
# make_gpu_notebook.py (gzip + base64: one long line freezes the editor).
import base64, gzip, json, os
ROOT = os.environ['P05_ROOT']
PACKED = gzip.decompress(base64.b64decode("""
...
oAgA
""")).decode()
FILES = json.loads(PACKED)
for p, text in FILES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, 'rewrites', p)), exist_ok=True)
    open(os.path.join(ROOT, 'rewrites', p), 'w').write(text)
print(len(FILES), 'files')


In [ ]:
%%bash
cd "$P05_ROOT"
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
$NVC --version | grep -v '^$' | head -1
python3 gpu_run.py --pb polybench --src rewrites --cc $NVC --omp-flags "-mp=gpu -gpu=cc$CC" \
  --threads $(nproc) --runs 3 --dataset LARGE_DATASET --csv polybench-t4.csv


In [ ]:
import csv, math, os
CSV = os.path.join(os.environ['P05_ROOT'], 'polybench-t4.csv')
rows = list(csv.DictReader(open(CSV)))
t = {(r['kernel'], r['variant']): float(r['time_s']) for r in rows if r['status'] == 'ok' and r['time_s']}
bad = [(r['kernel'], r['variant'], r['status'], r['correct']) for r in rows
       if r['status'] != 'ok' or r['correct'] not in ('', 'exact', 'tolerance')]
print('not ok / not correct:', bad or 'none')
for v in ('seq-mp', 'p05', 'p05-naive', 'p05-warm', 'p05-naive-warm'):
    xs = [t[(k, 'seq')] / t[(k, v)] for (k, vv) in t if vv == v and (k, 'seq') in t]
    if xs:
        print(f"{v:15} geomean speedup over seq {math.exp(sum(map(math.log, xs)) / len(xs)):.2f} ({len(xs)} kernels)")
try:
    from google.colab import files
    files.download(CSV)
except ImportError:  # Kaggle: files in /kaggle/working are the version's Output
    from IPython.display import FileLink, display
    print('CSV at', CSV)
    display(FileLink(os.path.relpath(CSV)))
